In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def audit_df(df, text_col, label_col=None):
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print("\nMissing values:")
    print(df.isna().sum())
    lengths = df[text_col].astype(str).str.len()
    print(lengths.describe())
    print("\nDuplicate text rows:", df.duplicated(subset=[text_col]).sum())
    if label_col is not None:
        print(df[label_col].value_counts(dropna=False))


def light_clean_df(df, text_col):
    df = df.copy()
    df[text_col] = df[text_col].apply(clean_text)
    df = df[df[text_col].str.len() > 0]
    df = df.drop_duplicates()  # Keep conflicting labels for the later task-level audit.
    return df.reset_index(drop=True)


In [ ]:
banking_test = pd.read_csv("data/raw/banking77_test.csv")
banking_train = pd.read_csv("data/raw/banking77_train.csv")
phrasebank = pd.read_csv("data/raw/all-data.csv", encoding = "latin-1",header=None,             # Correctly stops pandas from using your first data row as a header
    names=["label", "text"])  # this file has some non-UTF-8 chars, so specify encoding

frimi = pd.read_csv("data/raw/FriMi_FinTech_raw_reviews.csv")
keells = pd.read_csv("data/raw/Keells_Supermarket_raw_reviews.csv")
peoplespay = pd.read_csv("data/raw/Peoples_Pay_raw_reviews.csv")
combank = pd.read_csv("data/raw/ComBank_Digital_raw_reviews.csv")
flexpay = pd.read_csv("data/raw/FlexPay_BOC_raw_reviews.csv")
HNB = pd.read_csv("data/raw/HNB_DigitalBanking_raw_reviews.csv")
NDB = pd.read_csv("data/raw/NDB_MobileBanking_raw_reviews.csv")
NSB = pd.read_csv("data/raw/NSBPay_raw_reviews.csv")



In [ ]:
audit_df(banking_test, text_col="text", label_col="label")
audit_df(banking_train, text_col="text", label_col="label")


In [ ]:
audit_df(phrasebank, text_col="text", label_col="label")


In [ ]:
audit_df(frimi, text_col="review_text", label_col="star_rating")
audit_df(keells, text_col="review_text", label_col="star_rating")
audit_df(peoplespay, text_col="review_text", label_col="star_rating")
audit_df(combank, text_col="review_text", label_col="star_rating")
audit_df(flexpay, text_col="review_text", label_col="star_rating")
audit_df(NDB, text_col="review_text", label_col="star_rating")
audit_df(HNB, text_col="review_text", label_col="star_rating")
audit_df(NSB, text_col="review_text", label_col="star_rating")


In [ ]:
import unicodedata
clean_banking_test  = light_clean_df(banking_test, text_col="text")
clean_banking_train = light_clean_df(banking_train, text_col="text")
clean_phrasebank    = light_clean_df(phrasebank, text_col="text")

clean_frimi      = light_clean_df(frimi, text_col="review_text")
clean_keells     = light_clean_df(keells, text_col="review_text")
clean_peoplespay = light_clean_df(peoplespay, text_col="review_text")
clean_combank    = light_clean_df(combank, text_col="review_text")
clean_flexpay    = light_clean_df(flexpay, text_col="review_text")
clean_HNB      = light_clean_df(HNB, text_col="review_text")
clean_NSB      = light_clean_df(NSB, text_col="review_text")
clean_NDB     = light_clean_df(NDB, text_col="review_text")



In [ ]:
print("UNIQUE LABELS IN FINANCIAL PHRASEBANK")
print(clean_phrasebank["label"].unique())

print("SAMPLE OF UNIQUE INTENTS IN BANKING77")
print(list(clean_banking_train["label_text"].unique()))


In [ ]:
# 1. Preview Banking Datasets
print("CLEANED BANKING TEST PREVIEW")
display(clean_banking_test.head(3))

print("CLEANED BANKING TRAIN PREVIEW")
display(clean_banking_train.head(3))

# 2. Preview Phrasebank Dataset
print("CLEANED FINANCIAL PHRASEBANK PREVIEW")
display(clean_phrasebank.head(3))


In [ ]:

clean_banking_test.to_csv("data/processed/banking77_test_clean.csv", index=False)
clean_banking_train.to_csv("data/processed/banking77_train_clean.csv", index=False)

clean_phrasebank.to_csv("data/processed/financial_phrasebank_clean.csv", index=False)




In [ ]:
clean_bank_reviews = pd.concat(
    [
        clean_frimi,
        clean_keells,
        clean_peoplespay,
        clean_combank,
        clean_flexpay,
        clean_HNB,
        clean_NSB,
        clean_NDB,
    ],
    ignore_index=True,
)

if "normalized_text" not in clean_bank_reviews.columns:
    clean_bank_reviews["normalized_text"] = clean_bank_reviews["review_text"].apply(clean_text)

clean_bank_reviews = clean_bank_reviews.drop_duplicates(
    subset=["source_app", "normalized_text", "recorded_at"],
    keep="first",
).reset_index(drop=True)
clean_bank_reviews["source_row_id"] = range(1, len(clean_bank_reviews) + 1)

clean_bank_reviews.to_csv(
    "data/processed/scraped_reviews_clean_combined.csv",
    index=False,
)

print(
    "Saved cleaned bank reviews: "
    f"data/processed/scraped_reviews_combined.csv)"
)
print("Columns:", list(clean_bank_reviews.columns))